# V26 POC — Road overlap or road within N metres

This notebook evaluates the proposed `ROAD_TOO_CLOSE` rule against the ignored
real Kobo export in `notebooks/data`. It measures the shortest distance from
each plot **boundary** to the nearest eligible OpenStreetMap road, treats an
intersection as distance zero, and reports how many plots would be flagged for
a range of candidate thresholds `N`.

Privacy rules follow V24: raw rows, names, phone numbers, attachment URLs, and
exact plot coordinates are never displayed, plot identifiers in outputs are
one-way hashes, and every generated artifact stays below the fully ignored
`notebooks/data/v26_poc_output` directory. OSM way identifiers are public
reference data and are kept unhashed so a GIS reviewer can verify a result.
The notebook performs no database writes.

## Road source used by this POC

Production uses the Geofabrik Ethiopia extract (V26 D-3). That file is roughly
400 MB and covers the whole country, which is unnecessary for a feasibility
check on one collection area. This POC therefore fetches the **same OSM road
features, with the same `highway` classes and the same `osm_id` values**, from
the Overpass API for a padded bounding box around the plots.

The consequence is explicit and carried into the plan: this POC validates the
**measurement, threshold, and flag contract**, not the Geofabrik download,
checksum, unpack, filter, and atomic-activation pipeline. Those import stages
remain estimated engineering work with no POC evidence behind them.

## 1. Environment and configuration

Run from the repository root. Required packages are `requests`, `shapely`, and
`pyproj`. The eligible-class allow-list is the one proposed in the plan; every
other `highway` value (`residential`, `service`, `unclassified`, `track`,
`path`, `footway`, `bridleway`, ...) is excluded.

No production value of `N` exists yet, so the notebook evaluates a candidate
range instead of assuming one. `N_CANDIDATES` is the evidence product needs in
order to choose it.

In [ ]:
from pathlib import Path
import json
import statistics
import sys

import requests
from shapely.geometry import LineString, mapping
from shapely.ops import transform as shapely_transform
from shapely.strtree import STRtree

sys.path.insert(0, str(Path.cwd() / 'notebooks'))
from poc_common import (
    dataset_utm_crs,
    load_backend_helpers,
    load_records,
    poc_paths,
    project_polygon,
    utm_transformer,
    write_public_output,
)

PATHS = poc_paths('v26')
OUTPUT_DIR = PATHS['output_dir']
ROADS_CACHE_PATH = OUTPUT_DIR / 'osm_roads_bbox_private.json'
RESULT_PATH = OUTPUT_DIR / 'road_results_private.json'
AC_DEMO_PATH = OUTPUT_DIR / 'v26_acceptance_demo.json'

OVERPASS_ENDPOINTS = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.kumi.systems/api/interpreter',
    'https://overpass.osm.ch/api/interpreter',
]
OVERPASS_USER_AGENT = 'african-bamboo-dashboard-v26-poc/1.0'
ROAD_SOURCE = 'OpenStreetMap'
ROAD_SOURCE_PROVIDER = 'Overpass API (POC); Geofabrik extract (production)'
ROAD_RULE_VERSION = 'v1'

ELIGIBLE_OSM_HIGHWAYS = {
    'motorway', 'motorway_link',
    'trunk', 'trunk_link',
    'primary', 'primary_link',
    'secondary', 'secondary_link',
    'tertiary', 'tertiary_link',
}

BBOX_PADDING_DEGREES = 0.05
N_CANDIDATES = [10.0, 25.0, 50.0, 100.0, 200.0, 500.0]

print({
    'csv_present': PATHS['csv'].exists(),
    'cached_roads_present': ROADS_CACHE_PATH.exists(),
    'eligible_class_count': len(ELIGIBLE_OSM_HIGHWAYS),
    'threshold_candidates_m': N_CANDIDATES,
})

## 2. Load the real submissions through the production parser

In [ ]:
HELPERS = load_backend_helpers(PATHS['repo_root'])
records, valid_records = load_records(PATHS['csv'], HELPERS)

print({
    'submission_count': len(records),
    'valid_polygon_count': len(valid_records),
    'invalid_polygon_count': len(records) - len(valid_records),
    'raw_values_displayed': False,
})

## 3. Fetch eligible roads for the padded plot bounding box

The query bounding box is derived from the valid plots plus a padding halo and
is deliberately not printed. The regular expression on `highway` mirrors the
production allow-list, so the returned features are exactly those a filtered
Geofabrik import would retain for this area. The response is cached, so a
rerun performs no network request.

In [ ]:
all_coords = [coord for record in valid_records for coord in record['coords']]
private_bbox = HELPERS['compute_bbox'](all_coords)

CLASS_PATTERN = '^(motorway|trunk|primary|secondary|tertiary)(_link)?$'


def fetch_roads(destination):
    if destination.exists() and destination.stat().st_size > 0:
        return json.loads(destination.read_text()), 'cached'
    query = (
        '[out:json][timeout:180];'
        f'way["highway"~"{CLASS_PATTERN}"]'
        f'({private_bbox["min_lat"] - BBOX_PADDING_DEGREES},'
        f'{private_bbox["min_lon"] - BBOX_PADDING_DEGREES},'
        f'{private_bbox["max_lat"] + BBOX_PADDING_DEGREES},'
        f'{private_bbox["max_lon"] + BBOX_PADDING_DEGREES});'
        'out geom;'
    )
    errors = []
    for endpoint in OVERPASS_ENDPOINTS:
        try:
            response = requests.post(
                endpoint,
                data={'data': query},
                headers={'User-Agent': OVERPASS_USER_AGENT},
                timeout=300,
            )
            response.raise_for_status()
            payload = response.json()
        except (requests.RequestException, ValueError) as error:
            # Public Overpass mirrors rate-limit and time out under load.
            errors.append(f'{endpoint.split("/")[2]}: {type(error).__name__}')
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_text(json.dumps(payload))
        return payload, 'downloaded'
    raise RuntimeError(f'All Overpass mirrors failed: {errors}')


overpass_payload, roads_state = fetch_roads(ROADS_CACHE_PATH)
raw_ways = overpass_payload.get('elements', [])

roads = []
for way in raw_ways:
    highway_class = way.get('tags', {}).get('highway')
    if highway_class not in ELIGIBLE_OSM_HIGHWAYS:
        continue
    geometry = way.get('geometry') or []
    if len(geometry) < 2:
        continue
    roads.append({
        'osm_id': way['id'],
        'highway_class': highway_class,
        'line': LineString([(point['lon'], point['lat']) for point in geometry]),
    })

class_counts = {}
for road in roads:
    class_counts[road['highway_class']] = (
        class_counts.get(road['highway_class'], 0) + 1
    )

print({
    'roads_state': roads_state,
    'returned_ways': len(raw_ways),
    'eligible_ways': len(roads),
    'rejected_ways': len(raw_ways) - len(roads),
    'eligible_class_counts': class_counts,
    'bbox_printed': False,
})

## 4. Project once and build the spatial index

Distances must be metric, so both the plots and the roads are projected into
the collection area's UTM zone before any measurement. The `STRtree` index is
built once over the projected road lines — the production equivalent of the
per-process cached spatial index described in V26 D-3.

In [ ]:
UTM_CRS = dataset_utm_crs(valid_records)
TO_UTM = utm_transformer(UTM_CRS)

for road in roads:
    road['line_utm'] = shapely_transform(TO_UTM.transform, road['line'])

road_lines = [road['line_utm'] for road in roads]
road_index = STRtree(road_lines)

print({
    'projected_crs': UTM_CRS.to_string(),
    'indexed_roads': len(road_lines),
    'total_road_length_km': round(
        sum(line.length for line in road_lines) / 1000.0, 2
    ),
})

## 5. Boundary distance and overlap per plot

For each plot the notebook asks the index for the nearest eligible road and
then measures `boundary.distance(road)`. An intersecting road short-circuits to
distance `0.0` and `overlap = True`.

Nearest-neighbour search is used deliberately instead of a threshold-sized
envelope query. An envelope sized to `N` returns nothing for a plot that has no
road nearby, and "no road within N metres" is a **pass**, not a failed
measurement. Recording it as unavailable would put clean plots into the
unavailable state and hide them from Helen's pass/fail reading. Storing the
true distance also satisfies V26 D-5: `N` can change later without any
recalculation.

In [ ]:
def road_flag(distance_m, overlap, road_class, threshold_m):
    if not overlap and distance_m > threshold_m:
        return None
    if overlap:
        description = f'A {road_class} road overlaps this plot'
    else:
        description = (
            f'The nearest {road_class} road is {distance_m:.1f} m '
            'from the plot boundary'
        )
    return {
        'type': 'ROAD_TOO_CLOSE',
        'severity': 'warning',
        'note': (
            f'{description} (threshold: {threshold_m:g} m). '
            f'Dataset source: {ROAD_SOURCE}.'
        ),
    }


def measure_road(polygon_wgs84, road_list=None, index=None):
    """Return nearest eligible road distance, overlap, and identity.

    ``road_list``/``index`` default to the collection-area dataset built above.
    The synthetic demo passes its own pair so it can reuse this exact function
    at a different location.
    """
    road_list = roads if road_list is None else road_list
    index = road_index if index is None else index
    if not road_list:
        return None
    plot_utm = project_polygon(polygon_wgs84, TO_UTM)
    nearest = road_list[int(index.nearest(plot_utm))]
    overlap = plot_utm.intersects(nearest['line_utm'])
    distance = (
        0.0 if overlap else plot_utm.boundary.distance(nearest['line_utm'])
    )
    return {
        'distance_m': round(float(distance), 3),
        'overlap': bool(overlap),
        'nearest_osm_id': nearest['osm_id'],
        'nearest_highway_class': nearest['highway_class'],
    }


results = []
for record in valid_records:
    measurement = measure_road(record['polygon'])
    if measurement is None:
        results.append({
            'plot_ref': record['plot_ref'],
            'status': 'unavailable',
            'value': None,
            'details': {'reason': 'no_eligible_road_dataset_loaded'},
        })
        continue
    results.append({
        'plot_ref': record['plot_ref'],
        'status': 'complete',
        'value': measurement['distance_m'],
        'details': {
            'overlap': measurement['overlap'],
            'nearest_osm_id': measurement['nearest_osm_id'],
            'nearest_highway_class': measurement['nearest_highway_class'],
        },
    })

complete_results = [r for r in results if r['status'] == 'complete']
print({
    'analysed_plots': len(results),
    'complete': len(complete_results),
    'unavailable': len(results) - len(complete_results),
})

## 6. Threshold sensitivity — the evidence for choosing N

`N` is an open question in the plan. The table below shows how many of the real
plots each candidate threshold would place in Helen's review queue, which is
the operational cost of the choice.

In [ ]:
distances = [r['value'] for r in complete_results]
overlaps = [r for r in complete_results if r['details']['overlap']]

sensitivity = {}
for threshold in N_CANDIDATES:
    flagged = [
        r for r in complete_results
        if r['details']['overlap'] or r['value'] <= threshold
    ]
    sensitivity[f'{threshold:g} m'] = {
        'flagged_plots': len(flagged),
        'flagged_percent': round(
            len(flagged) / len(complete_results) * 100.0, 1
        ),
    }

summary = {
    'valid_input_polygons': len(valid_records),
    'completed_calculations': len(complete_results),
    'plots_overlapping_a_road': len(overlaps),
    'min_distance_m': round(min(distances), 2),
    'median_distance_m': round(statistics.median(distances), 2),
    'max_distance_m': round(max(distances), 2),
    'nearest_class_counts': {
        highway_class: sum(
            1 for r in complete_results
            if r['details']['nearest_highway_class'] == highway_class
        )
        for highway_class in sorted({
            r['details']['nearest_highway_class'] for r in complete_results
        })
    },
    'threshold_sensitivity': sensitivity,
}
print(json.dumps(summary, indent=2))

## 7. Boundary-contract demo using existing records

The plan states that `N` is inclusive: a plot exactly `N` metres away is
flagged and `N + 0.01` is not. Each scenario reuses an existing hashed plot
record and its real measured distance, but substitutes a controlled evaluation
value. Controlled values are labelled and never replace the observed
measurement.

In [ ]:
if len(complete_results) < 3:
    raise RuntimeError('At least three complete results are required')

DEMO_THRESHOLD_M = 100.0
controlled_scenarios = [
    ('overlapping_road', complete_results[0], 0.0, True),
    ('exactly_at_threshold', complete_results[1], DEMO_THRESHOLD_M, False),
    ('just_outside_threshold', complete_results[2], 100.01, False),
]

acceptance_rows = []
for scenario, real_result, controlled_value, controlled_overlap in (
    controlled_scenarios
):
    road_class = real_result['details']['nearest_highway_class']
    flag = road_flag(
        controlled_value, controlled_overlap, road_class, DEMO_THRESHOLD_M
    )
    acceptance_rows.append({
        'scenario': scenario,
        'uses_existing_hashed_plot': True,
        'plot_ref': real_result['plot_ref'],
        'observed_distance_m': real_result['value'],
        'controlled_demo_distance_m': controlled_value,
        'controlled_demo_overlap': controlled_overlap,
        'controlled_demo_input': True,
        'flagged_for_review': bool(flag),
        'flagged_reason': [flag] if flag else [],
        'geospatial_metrics': {
            'road': {
                'status': 'complete',
                'value': controlled_value,
                'unit': 'metres',
                'source': ROAD_SOURCE,
                'source_provider': ROAD_SOURCE_PROVIDER,
                'threshold': {
                    'operator': '<=',
                    'value': DEMO_THRESHOLD_M,
                    'unit': 'metres',
                    'rule_version': ROAD_RULE_VERSION,
                },
                'details': {
                    'overlap': controlled_overlap,
                    'nearest_highway_class': road_class,
                    'nearest_osm_id': real_result['details']['nearest_osm_id'],
                },
            }
        },
    })

AC_DEMO_PATH.write_text(json.dumps(acceptance_rows, indent=2))
print([
    (row['scenario'], row['flagged_for_review'])
    for row in acceptance_rows
])

## 8. Threshold-only reevaluation without recalculating distances

V26 D-5 requires that changing `N` reevaluates flags from the stored measured
distance and does not recalculate geometry or reread road data. The check below
demonstrates that property directly: flags are derived twice from the same
stored values with two different thresholds, and no measurement function is
called.

In [ ]:
def flags_for_threshold(stored_results, threshold_m):
    flags = {}
    for result in stored_results:
        flag = road_flag(
            result['value'],
            result['details']['overlap'],
            result['details']['nearest_highway_class'],
            threshold_m,
        )
        if flag:
            flags[result['plot_ref']] = flag
    return flags

low_threshold_flags = flags_for_threshold(complete_results, 25.0)
high_threshold_flags = flags_for_threshold(complete_results, 200.0)

print({
    'flagged_at_25m': len(low_threshold_flags),
    'flagged_at_200m': len(high_threshold_flags),
    'is_monotonic_superset': set(low_threshold_flags).issubset(
        set(high_threshold_flags)
    ),
    'distances_recalculated': False,
})

## 9. Persist the POC artifacts

Output is split by sensitivity, and V26 is the strictest of the four POCs.

`notebooks/outputs/` is **tracked by git** and receives the aggregate summary,
the threshold sensitivity table, and the boundary demo.

`notebooks/data/v26_poc_output/` stays **fully ignored**. Its per-plot rows pair
`nearest_osm_id` with an exact distance, which pins a plot to a ring around a
specific mappable OpenStreetMap way; 166 such rows reconstruct the collection
area. The cached Overpass response is bounding-box derived and discloses the
area directly. `write_public_output` refuses to write `nearest_osm_id`, so the
separation is enforced rather than merely intended.

In [ ]:
payload = {
    'summary': summary,
    'dataset': {
        'source': ROAD_SOURCE,
        'source_provider': ROAD_SOURCE_PROVIDER,
        'eligible_classes': sorted(ELIGIBLE_OSM_HIGHWAYS),
        'observed_eligible_ways': len(roads),
        'production_source_note': (
            'Production imports the Geofabrik Ethiopia extract; this POC '
            'used a bounding-box Overpass query of the same OSM features.'
        ),
    },
    'results': [
        {
            'plot_ref': result['plot_ref'],
            'geospatial_metrics': {
                'road': {
                    'status': result['status'],
                    'value': result['value'],
                    'unit': 'metres',
                    'source': ROAD_SOURCE,
                    'source_provider': ROAD_SOURCE_PROVIDER,
                    'details': result['details'],
                }
            },
        }
        for result in results
    ],
}
RESULT_PATH.write_text(json.dumps(payload, indent=2))

public = write_public_output('v26_road_summary.json', {
    'poc': 'v26',
    'metric': 'road',
    'notebook': 'notebooks/v26_road_proximity_poc.ipynb',
    'dataset': payload['dataset'],
    'summary': summary,
    'boundary_demo': [
        {
            'scenario': row['scenario'],
            'controlled_demo_distance_m': row['controlled_demo_distance_m'],
            'controlled_demo_overlap': row['controlled_demo_overlap'],
            'flagged_for_review': row['flagged_for_review'],
            'note': (
                row['flagged_reason'][0]['note']
                if row['flagged_reason'] else None
            ),
        }
        for row in acceptance_rows
    ],
    'threshold_only_reevaluation': {
        'flagged_at_25m': len(low_threshold_flags),
        'flagged_at_200m': len(high_threshold_flags),
        'is_monotonic_superset': set(low_threshold_flags).issubset(
            set(high_threshold_flags)
        ),
        'distances_recalculated': False,
    },
})

print({
    'ignored_private_files': [
        RESULT_PATH.name, AC_DEMO_PATH.name, ROADS_CACHE_PATH.name,
    ],
    'tracked_public_file': public['file'],
})

## 11. Synthetic example — what a flagged plot looks like

No real plot overlaps a road, so the overlap branch has no evidence behind it
beyond the controlled demo. This section hand-draws a 40 m example plot
**centred on a vertex of a real trunk road far from the collection area**,
which guarantees an intersection, and measures it with `measure_road` above.

Two properties make this safe to commit: the polygon is fabricated, and the
location is a public highway rather than a farm, so the map discloses nothing
about where African Bamboo works.

In [ ]:
from poc_common import DEMO_LOCATIONS, preview_map, synthetic_square

DEMO_SIDE_M = 40.0
DEMO_PAD_DEGREES = 0.02
demo_lon, demo_lat, demo_place = DEMO_LOCATIONS['trunk_road']

demo_query = (
    '[out:json][timeout:180];'
    f'way["highway"~"{CLASS_PATTERN}"]'
    f'({demo_lat - DEMO_PAD_DEGREES},{demo_lon - DEMO_PAD_DEGREES},'
    f'{demo_lat + DEMO_PAD_DEGREES},{demo_lon + DEMO_PAD_DEGREES});'
    'out geom;'
)
demo_payload = None
for endpoint in OVERPASS_ENDPOINTS:
    try:
        response = requests.post(
            endpoint,
            data={'data': demo_query},
            headers={'User-Agent': OVERPASS_USER_AGENT},
            timeout=300,
        )
        response.raise_for_status()
        demo_payload = response.json()
        break
    except (requests.RequestException, ValueError):
        continue
if demo_payload is None:
    raise RuntimeError('All Overpass mirrors failed for the demo location')

demo_roads = []
for way in demo_payload.get('elements', []):
    highway_class = way.get('tags', {}).get('highway')
    geometry = way.get('geometry') or []
    if highway_class not in ELIGIBLE_OSM_HIGHWAYS or len(geometry) < 2:
        continue
    line = LineString([(p['lon'], p['lat']) for p in geometry])
    demo_roads.append({
        'osm_id': way['id'],
        'highway_class': highway_class,
        'line': line,
        'line_utm': shapely_transform(TO_UTM.transform, line),
    })

demo_index = STRtree([road['line_utm'] for road in demo_roads])

# Centre the example plot on a mid-way vertex so it certainly overlaps.
demo_road = demo_roads[0]
vertex = list(demo_road['line'].coords)[len(demo_road['line'].coords) // 2]
demo_plot = synthetic_square(vertex[0], vertex[1], DEMO_SIDE_M)

demo_measurement = measure_road(demo_plot, demo_roads, demo_index)
demo_flag = road_flag(
    demo_measurement['distance_m'],
    demo_measurement['overlap'],
    demo_measurement['nearest_highway_class'],
    DEMO_THRESHOLD_M,
)
print(json.dumps({
    'location': demo_place,
    'eligible_roads_found': len(demo_roads),
    'distance_m': demo_measurement['distance_m'],
    'overlap': demo_measurement['overlap'],
    'nearest_highway_class': demo_measurement['nearest_highway_class'],
    'threshold_m': DEMO_THRESHOLD_M,
    'flagged': bool(demo_flag),
    'note': demo_flag['note'] if demo_flag else None,
}, indent=2))

The map below is the visual check. A red outline means the rule
fired; click the polygon for the measured values. Run
`pip install -r notebooks/requirements.txt` if folium is missing.

In [ ]:
preview_map(
    demo_plot,
    flagged=bool(demo_flag),
    title=f'V26 example - {demo_place}',
    rows={
        'Nearest road': demo_measurement['nearest_highway_class'],
        'Distance': f"{demo_measurement['distance_m']:.1f} m",
        'Overlap': demo_measurement['overlap'],
        'Threshold': f'{DEMO_THRESHOLD_M:g} m',
        'Result': 'ROAD_TOO_CLOSE' if demo_flag else 'pass',
    },
    lines=[
        (road['highway_class'], road['line'])
        for road in demo_roads[:12]
    ],
    zoom=17,
)

## 10. Review checklist and interpretation

The POC is technically successful when every valid geoshape parses, eligible
road classes are filtered as specified, distances are measured in a metric CRS
from the polygon boundary, overlap resolves to distance zero, the inclusive
`<= N` contract holds, and changing `N` reevaluates flags without recalculating
any distance.

What the POC does **not** cover, and what therefore stays as estimated work
with no evidence behind it:

- Geofabrik download, checksum verification, safe unpack, and class filtering.
- Versioned dataset registration and atomic activation.
- Six-month refresh scheduling, failure recovery, and delta reporting.
- Behaviour at national scale; this run indexes one collection area only.

Two product decisions are informed directly by the printed summary:

1. The value of `N`. `threshold_sensitivity` shows the review-queue cost of
   each candidate against real plots.
2. The eligible-class allow-list. `nearest_class_counts` shows which classes
   actually drive the result here; if the collection area contains only one
   class, the broader allow-list is untested in practice and should be
   reviewed against a wider sample before rollout.

For an independent QGIS spot check, load the ignored
`osm_roads_bbox_private.json` and compare a few hashed records. Do not copy the
real polygons outside the ignored POC directory.

Attribution: road data © OpenStreetMap contributors, available under the Open
Database License (ODbL).